# 02 — Modeling walkthrough

End-to-end run of the LoanGuard training pipeline on a small sample.

1. Load + label
2. Time-based split
3. Feature engineering
4. Train XGBoost + LightGBM (baseline)
5. Train stacking ensemble
6. Evaluate on the test set

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.data import LendingClubLoader, build_fraud_labels, LabelConfig
from src.data.splitter import time_based_split
from src.features import FeatureBuilder
from src.models import XGBFraudModel, LGBMFraudModel, IsolationForestFraudModel, StackingFraudEnsemble
from src.evaluation import binary_classification_metrics

In [ ]:
# 1. Load + label
df = LendingClubLoader(
    raw_path='../data/raw/accepted_2007_to_2018Q4.csv',
    sample_size=30_000,
).load()
df = build_fraud_labels(df, LabelConfig())
print(df.is_fraud.value_counts(normalize=True))

In [ ]:
# 2. Split
train, val, test = time_based_split(df, 'issue_d', 0.15, 0.15)
drop = ['is_fraud', 'rule_fpd', 'rule_income_anomaly', 'rule_debt_inconsist',
        'rule_address_ring', 'n_anomalies', 'loan_status', 'last_pymnt_d']
X_train_raw = train.drop(columns=drop); y_train = train.is_fraud
X_val_raw   = val.drop(columns=drop);   y_val   = val.is_fraud
X_test_raw  = test.drop(columns=drop);  y_test  = test.is_fraud

In [ ]:
# 3. Features
builder = FeatureBuilder(use_velocity=False, use_graph=False)  # off for speed in notebook
X_train = builder.fit_transform(X_train_raw, y_train)
X_val   = builder.transform(X_val_raw)
X_test  = builder.transform(X_test_raw)
X_train.head()

In [ ]:
# 4. Baseline: XGB and LGBM
xgb = XGBFraudModel(params={'n_estimators': 300, 'max_depth': 5}).fit(X_train, y_train, eval_set=[(X_val, y_val)])
lgb = LGBMFraudModel(params={'n_estimators': 300, 'num_leaves': 31}).fit(X_train, y_train, eval_set=[(X_val, y_val)])
iso = IsolationForestFraudModel(params={'n_estimators': 100}).fit(X_train)

for name, m in [('xgb', xgb), ('lgb', lgb), ('iso', iso)]:
    p = m.predict_proba(X_test)
    print(name, binary_classification_metrics(y_test, p))

In [ ]:
# 5. Stacking
ens = StackingFraudEnsemble(base_models=[xgb, lgb, iso], n_folds=3).fit(
    X_train, y_train, X_val=X_val, y_val=y_val
)
binary_classification_metrics(y_test, ens.predict_proba(X_test))